# Research source snapshot
Outputs and execution metadata have been removed. Read the repository README before running. These exploratory notebooks are not validated end to end. Only load trusted local pickle files.


In [ ]:
import os
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("RESEARCH_DATA_DIR", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("RESEARCH_OUTPUT_DIR", str(REPOSITORY_ROOT / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import geopandas as gpd

file_2011 = str(DATA_ROOT / 'gwra2011.geojson')

file_2013 = str(DATA_ROOT / 'gwra2013.geojson')

file_2017 = str(DATA_ROOT / 'gwra2017.geojson')

file_2020 = str(DATA_ROOT / 'gwra2020.geojson')

In [ ]:
gdf_2011 = gpd.read_file(file_2011)

In [ ]:
gdf_2011

In [ ]:
for column in gdf_2011.columns:
  print(column)

In [ ]:
print(gdf_2011.dtypes)

In [ ]:
gdf_2013 = gpd.read_file(file_2013)

In [ ]:
gdf_2013

In [ ]:
print(gdf_2013.dtypes)

In [ ]:
gdf_2017 = gpd.read_file(file_2017)

In [ ]:
gdf_2017

In [ ]:
print(gdf_2017.dtypes)

In [ ]:
gdf_2020 = gpd.read_file(file_2020)

In [ ]:
gdf_2020

In [ ]:
print(gdf_2020.dtypes)

In [ ]:
import pandas as pd

# Path to your CSV file
file_path = str(DATA_ROOT / 'shallow_aquifers_csv_export.csv')

# Read the CSV file
df = pd.read_csv(file_path)

# Display basic information about the dataframe
print(f"Shape of the dataframe: {df.shape}")
print("\nFirst few rows:")
display(df.head())

# Display column names
print("\nColumns in the dataframe:")
print(df.columns.tolist())

# Summary statistics
print("\nSummary statistics:")
display(df.describe())


In [ ]:
print(df['microwatershed_id'].value_counts())

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkt

import geopandas as gpd
from shapely import wkt
import json
from shapely.geometry import shape


# --- Step A: Convert df to a GeoDataFrame properly ---
if isinstance(df['.geo'].iloc[0], str):
    df['geometry'] = df['.geo'].apply(lambda x: shape(json.loads(x)))

gdf_microwatersheds20 = gpd.GeoDataFrame(df, geometry='geometry')
gdf_microwatersheds20.crs = "EPSG:4326"

# --- Step B: Reproject microwatersheds if necessary ---
print("gdf_2020 CRS:", gdf_2020.crs)
if gdf_2020.crs != "EPSG:4326":
    gdf_microwatersheds20 = gdf_microwatersheds20.to_crs(gdf_2020.crs)
    print("Reprojected microwatersheds to match blocks CRS:", gdf_microwatersheds20.crs)

# --- Step C: Perform spatial join ---
joined_gdf20 = gpd.sjoin(gdf_microwatersheds20, gdf_2020, how='left', predicate='intersects')
print("\nJoin results (before filtering duplicates):")
print(joined_gdf20[['microwatershed_id', 'block', 'class', 'sgw_dev_pe']].head())

# --- Step D: Select one block per microwatershed ---
# Option 1: Drop duplicate microwatershed_ids (keeps the first occurrence)
unique_joined_gdf20 = joined_gdf20.drop_duplicates(subset='microwatershed_id')

# Option 2 (alternative): Select the block with the maximum overlap (comment out the above line to use this)
# joined_gdf20['intersection_area'] = joined_gdf20.geometry.intersection(gdf_2020.unary_union).area
# idx = joined_gdf20.groupby('microwatershed_id')['intersection_area'].idxmax()
# unique_joined_gdf20 = joined_gdf20.loc[idx].reset_index(drop=True)

print("\nJoin results (one block per microwatershed):")
print(unique_joined_gdf20[['microwatershed_id', 'block', 'class', 'sgw_dev_pe']].head())

# Count microwatersheds by block
block_counts20 = unique_joined_gdf20.groupby('block').size().reset_index(name='microwatershed_count')
print("\nNumber of microwatersheds in each block:")
print(block_counts20)

# --- Step D: Read India State Boundary and Plot ---
india_url = str(DATA_ROOT / "Indian_states_boundary.geojson")
india_state_boundary_map = gpd.read_file(india_url)

# Ensure both GeoDataFrames use the same CRS (here, we choose target_crs as gdf_2020.crs)
target_crs = gdf_2020.crs
if india_state_boundary_map.crs != target_crs:
    india_state_boundary_map = india_state_boundary_map.to_crs(target_crs)

# Extract unique blocks with geometry (if needed, use gdf_2020 for block geometries)
unique_blocks_gdf = joined_gdf20[['block', 'geometry']].drop_duplicates(subset='block').copy()

# Plot the blocks on top of the India state boundaries
fig, ax = plt.subplots(figsize=(12, 12))
india_state_boundary_map.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.5)
unique_blocks_gdf.plot(
    ax=ax,
    column='block',
    cmap='Set1',
    edgecolor='darkred',
    linewidth=1,
    alpha=0.7,
    #legend=True
)
ax.set_title("Blocks Projected on India State Boundaries", fontsize=16)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
# import contextily as ctx
from shapely import wkt

In [ ]:
# Step 7: Create a list of blocks with their geometries
# Dissolve gdf_2020 by block to get one geometry per block
blocks_with_geom = gdf_2020.dissolve(by='block').reset_index()

# Merge with counts
blocks_with_data = blocks_with_geom.merge(block_counts, on='block', how='left')
blocks_with_data['microwatershed_count'] = blocks_with_data['microwatershed_count'].fillna(0)

# Print the list of blocks with their count of microwatersheds
print("\nList of blocks with microwatershed counts:")
for idx, row in blocks_with_data.iterrows():
    print(f"Block: {row['block']}, Microwatersheds: {int(row['microwatershed_count'])}")

# Step 8: Visualize on India map
# Read India state boundaries
india_url = str(DATA_ROOT / "Indian_states_boundary.geojson")
india_states = gpd.read_file(india_url)

import geopandas as gpd
import matplotlib.pyplot as plt

# --- Step 1. Extract unique blocks with their geometries ---
# Assuming joined_gdf contains a 'block' column and a 'geometry' column.
# If every microwatershed carries the blocks geometry, we might only need one representative record per block.
unique_blocks_gdf = joined_gdf[['block', 'geometry']].drop_duplicates(subset='block').copy()

print("Unique blocks with geometries:")
print(unique_blocks_gdf)

# --- Step 2. (Optional) Reproject the blocks to match India state boundaries CRS ---
# Check CRS of the India boundaries
print("India state boundary map CRS:", india_states.crs)

# Make sure both GeoDataFrames use the same CRS; here we set both to WGS84 if needed.
target_crs = "EPSG:4326"
if unique_blocks_gdf.crs != target_crs:
    unique_blocks_gdf = unique_blocks_gdf.to_crs(target_crs)
if india_states.crs != target_crs:
    india_state_boundary_map = india_states.to_crs(target_crs)

# --- Step 3. Plot the blocks on top of the India state boundaries ---
fig, ax = plt.subplots(figsize=(12, 12))

# Plot India boundaries first as the background map
india_states.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.5)

# Plot blocks with a distinct color; if the blocks are polygons, a colormap is used, and if they are points, adjust marker properties.
unique_blocks_gdf.plot(
    ax=ax,
    column='block',
    cmap='Set1',
    edgecolor='darkred',
    linewidth=1,
    alpha=0.7,

)

# Optionally, add a title and labels
ax.set_title("Blocks Projected on India State Boundaries", fontsize=16)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)

plt.show()


In [ ]:
# Step 3: Convert to GeoDataFrame based on what we find
if isinstance(df['geometry'].iloc[0], str):
    # If it's a string but not in WKT format (might be GeoJSON or something else)
    # Try to explicitly use GeoPandas' read_file instead
    gdf_microwatersheds17 = gpd.read_file(file_path)
else:
    # If we're dealing with a pandas DataFrame that has geometry objects already
    gdf_microwatersheds17 = gpd.GeoDataFrame(df, geometry='geometry')

# Step 4: Set the CRS
gdf_microwatersheds17.crs = "EPSG:4326"  # WGS84 is a common default

# Step 5: Check if gdf_2020 has a different CRS and reproject if needed
print("gdf_2020 CRS:", gdf_2017.crs)
if gdf_2017.crs != "EPSG:4326":
    gdf_microwatersheds17 = gdf_microwatersheds17.to_crs(gdf_2017.crs)
    print("Reprojected microwatersheds to match blocks CRS:", gdf_microwatersheds17.crs)

# Step 6: Perform spatial join
joined_gdf17 = gpd.sjoin(gdf_microwatersheds17, gdf_2017, how='left', predicate='intersects')
# Step 7: Show results
print("\nJoin results:")
print(joined_gdf17[['microwatershed_id', 'block','class','sgw_dev_pe']].head())

unique_joined_gdf17 = joined_gdf17.drop_duplicates(subset='microwatershed_id')

print("\nJoin results (one block per microwatershed):")
print(unique_joined_gdf17[['microwatershed_id', 'block', 'class', 'sgw_dev_pe']].head())


# Count microwatersheds by block
block_counts17 = unique_joined_gdf17.groupby('block').size().reset_index(name='microwatershed_count')
print("\nNumber of microwatersheds in each block:")
print(block_counts17)

In [ ]:
# Step 3: Convert to GeoDataFrame based on what we find
if isinstance(df['geometry'].iloc[0], str):
    # If it's a string but not in WKT format (might be GeoJSON or something else)
    # Try to explicitly use GeoPandas' read_file instead
    gdf_microwatersheds13 = gpd.read_file(file_path)
else:
    # If we're dealing with a pandas DataFrame that has geometry objects already
    gdf_microwatersheds13 = gpd.GeoDataFrame(df, geometry='geometry')

# Step 4: Set the CRS
gdf_microwatersheds13.crs = "EPSG:4326"  # WGS84 is a common default

# Step 5: Check if gdf_2020 has a different CRS and reproject if needed
print("gdf_2013 CRS:", gdf_2013.crs)
if gdf_2013.crs != "EPSG:4326":
    gdf_microwatersheds13 = gdf_microwatersheds13.to_crs(gdf_2013.crs)
    print("Reprojected microwatersheds to match blocks CRS:", gdf_microwatersheds13.crs)

# Step 6: Perform spatial join
joined_gdf13 = gpd.sjoin(gdf_microwatersheds13, gdf_2013, how='left', predicate='intersects')
# Step 7: Show results
print("\nJoin results:")
print(joined_gdf13[['microwatershed_id', 'block','class','sgw_dev_pe']].head())

unique_joined_gdf13 = joined_gdf13.drop_duplicates(subset='microwatershed_id')

# Option 2 (alternative): Select the block with the maximum overlap (comment out the above line to use this)
# joined_gdf20['intersection_area'] = joined_gdf20.geometry.intersection(gdf_2020.unary_union).area
# idx = joined_gdf20.groupby('microwatershed_id')['intersection_area'].idxmax()
# unique_joined_gdf20 = joined_gdf20.loc[idx].reset_index(drop=True)

print("\nJoin results (one block per microwatershed):")
print(unique_joined_gdf13[['microwatershed_id', 'block', 'class', 'sgw_dev_pe']].head())

# Count microwatersheds by block
block_counts13 = unique_joined_gdf13.groupby('block').size().reset_index(name='microwatershed_count')
print("\nNumber of microwatersheds in each block:")
print(block_counts13)

# Count microwatersheds by block
block_counts13 = unique_joined_gdf13.groupby('block').size().reset_index(name='microwatershed_count')
print("\nNumber of microwatersheds in each block:")
print(block_counts13)

In [ ]:
import pandas as pd

In [ ]:
# --- Step 1: Create copies and rename columns for 2013 ---
df_13 = unique_joined_gdf13[['microwatershed_id', 'class', 'sgw_dev_pe']].copy()
df_13 = df_13.rename(columns={
    'class': 'class_2013',
    'sgw_dev_pe': 'sgw_dev_pe_2013'
})

# --- Step 2: Create copies and rename columns for 2017 ---
df_17 = unique_joined_gdf17[['microwatershed_id', 'class', 'sgw_dev_pe']].copy()
df_17 = df_17.rename(columns={
    'class': 'class_2017',
    'sgw_dev_pe': 'sgw_dev_pe_2017'
})

# --- Step 3: Create copies and rename columns for 2020 ---
df_20 = unique_joined_gdf20[['microwatershed_id', 'class', 'sgw_dev_pe']].copy()
df_20 = df_20.rename(columns={
    'class': 'class_2020',
    'sgw_dev_pe': 'sgw_dev_pe_2020'
})

# --- Step 4: Merge the dataframes on microwatershed_id ---
# First, merge the 2013 and 2017 dataframes
merged_temp = pd.merge(df_13, df_17, on='microwatershed_id', how='outer')

# Then merge the result with the 2020 dataframe
merged_df = pd.merge(merged_temp, df_20, on='microwatershed_id', how='outer')

# --- Optional: Inspect the resulting dataframe ---
print(merged_df.head())

In [ ]:
# Calculate the slopes between the benchmark years
merged_df['slope_13_17'] = (merged_df['sgw_dev_pe_2017'] - merged_df['sgw_dev_pe_2013']) / 4.0
merged_df['slope_17_20'] = (merged_df['sgw_dev_pe_2020'] - merged_df['sgw_dev_pe_2017']) / 3.0

# ----- For Years 2014, 2015, 2016 (using 2013 as the base) -----
# For 2014: one year after 2013
merged_df['sgw_dev_pe_2014'] = merged_df['sgw_dev_pe_2013'] + merged_df['slope_13_17'] * 1
# For 2015: two years after 2013
merged_df['sgw_dev_pe_2015'] = merged_df['sgw_dev_pe_2013'] + merged_df['slope_13_17'] * 2
# For 2016: three years after 2013
merged_df['sgw_dev_pe_2016'] = merged_df['sgw_dev_pe_2013'] + merged_df['slope_13_17'] * 3

# ----- For Years 2018, 2019 (using 2017 as the base) -----
# For 2018: one year after 2017
merged_df['sgw_dev_pe_2018'] = merged_df['sgw_dev_pe_2017'] + merged_df['slope_17_20'] * 1
# For 2019: two years after 2017
merged_df['sgw_dev_pe_2019'] = merged_df['sgw_dev_pe_2017'] + merged_df['slope_17_20'] * 2

# ----- For Years 2021, 2022 (using 2020 as the base) -----
# For 2021: one year after 2020
merged_df['sgw_dev_pe_2021'] = merged_df['sgw_dev_pe_2020'] + merged_df['slope_17_20'] * 1
# For 2022: two years after 2020
merged_df['sgw_dev_pe_2022'] = merged_df['sgw_dev_pe_2020'] + merged_df['slope_17_20'] * 2

# Optional: Display the calculated values for inspection
columns_to_show = [
    'microwatershed_id',
    'sgw_dev_pe_2013', 'sgw_dev_pe_2014', 'sgw_dev_pe_2015', 'sgw_dev_pe_2016',
    'sgw_dev_pe_2017', 'sgw_dev_pe_2018', 'sgw_dev_pe_2019',
    'sgw_dev_pe_2020', 'sgw_dev_pe_2021', 'sgw_dev_pe_2022'
]
print(merged_df[columns_to_show].head())


In [ ]:
merged_df

In [ ]:
# List of columns you want to keep
columns_to_keep = ['microwatershed_id'] + [col for col in merged_df.columns if col.startswith('sgw_dev_pe_')]

# Create the new dataframe
soge_deep = merged_df[columns_to_keep]

# Save it to a CSV file (optional)
soge_deep.to_csv(str(OUTPUT_ROOT / 'soge_shallow.csv'), index=False)